# Chapitre 14 - Exercices 14.7 à 14.12

Composantes principales, problèmes de Procrustes et formes moyennes, positionnement
multidimensionnel classique, ACP parcimonieuse.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch14_non_supervise as ns
set_style()

## Un lemme utile : maximiser une trace sous contrainte d'orthonormalité

Plusieurs exercices utilisent le résultat suivant (inégalité de Ky Fan).

**Lemme.** Soit $\mathbf{S}$ une matrice $n \times n$ symétrique de valeurs propres $\lambda_1 \ge
\dots \ge \lambda_n$ et de vecteurs propres orthonormés $e_1, \dots, e_n$. Pour toute matrice
$\mathbf{M}$ ($n \times q$) telle que $\mathbf{M}^T\mathbf{M} = \mathbf{I}_q$,

$$\operatorname{tr}(\mathbf{M}^T\mathbf{S}\mathbf{M}) \le \lambda_1 + \dots + \lambda_q ,$$

avec égalité pour $\mathbf{M} = (e_1, \dots, e_q)$.

*Preuve.* On écrit $\mathbf{S} = \sum_j \lambda_j e_j e_j^T$, d'où
$\operatorname{tr}(\mathbf{M}^T\mathbf{S}\mathbf{M}) = \sum_j \lambda_j \operatorname{tr}(\mathbf{M}^Te_je_j^T\mathbf{M})
= \sum_j \lambda_j w_j$ avec $w_j = \|\mathbf{M}^Te_j\|^2$.

* $w_j \ge 0$, et $w_j = e_j^T\mathbf{M}\mathbf{M}^Te_j \le \|e_j\|^2 = 1$ car $\mathbf{M}\mathbf{M}^T$ est
  un projecteur orthogonal (il est symétrique et $(\mathbf{M}\mathbf{M}^T)^2 = \mathbf{M}\mathbf{M}^T$).
* $\sum_j w_j = \operatorname{tr}(\mathbf{M}^T(\sum_j e_je_j^T)\mathbf{M}) = \operatorname{tr}(\mathbf{M}^T\mathbf{M}) = q$.

Maximiser $\sum_j \lambda_j w_j$ sous $0 \le w_j \le 1$, $\sum_j w_j = q$ revient à mettre tout le
poids sur les $q$ plus grands $\lambda_j$ : le maximum vaut $\lambda_1 + \dots + \lambda_q$. Il est
atteint par $\mathbf{M} = (e_1, \dots, e_q)$, pour qui $w_1 = \dots = w_q = 1$. $\square$

In [ ]:
rng = np.random.RandomState(0)
A = rng.normal(size=(6, 6))
S = A + A.T
lam = np.sort(np.linalg.eigvalsh(S))[::-1]
essais = [np.trace(Q.T @ S @ Q) for Q in (np.linalg.qr(rng.normal(size=(6, 2)))[0] for _ in range(2000))]
print("max sur 2000 M aléatoires : %.4f  <=  lambda_1 + lambda_2 = %.4f" % (max(essais), lam[:2].sum()))

## Exercice 14.7 : Composantes principales : estimation de $\mu$ et des $\lambda_i$

**Énoncé.** Établir (14.51) et (14.52) de la section 14.5.1 : dans le problème
$\min_{\mu, \{\lambda_i\}, \mathbf{V}_q} \sum_i \|x_i - \mu - \mathbf{V}_q\lambda_i\|^2$ (14.50), où
$\mathbf{V}_q$ ($p \times q$) a des colonnes orthonormées, on peut prendre $\hat\mu = \bar x$ et
$\hat\lambda_i = \mathbf{V}_q^T(x_i - \bar x)$. Montrer que $\hat\mu$ n'est pas unique et
caractériser la famille des solutions équivalentes.

### Étape 1 : $\lambda_i$ à $\mu$ et $\mathbf{V}_q$ fixés

Les termes de la somme sont indépendants : pour chaque $i$, on minimise $\|y_i - \mathbf{V}_q\lambda_i\|^2$
avec $y_i = x_i - \mu$. C'est une régression par moindres carrés de $y_i$ sur les colonnes de
$\mathbf{V}_q$ ; les équations normales donnent $\mathbf{V}_q^T\mathbf{V}_q\lambda_i = \mathbf{V}_q^Ty_i$, et
comme $\mathbf{V}_q^T\mathbf{V}_q = \mathbf{I}_q$ :

$$\hat\lambda_i = \mathbf{V}_q^T(x_i - \mu).$$

Le résidu est $(\mathbf{I} - \mathbf{P})(x_i - \mu)$ avec $\mathbf{P} = \mathbf{V}_q\mathbf{V}_q^T$, projecteur
orthogonal sur le sous-espace engendré par les colonnes de $\mathbf{V}_q$.

### Étape 2 : $\mu$ à $\mathbf{V}_q$ fixé

Il reste à minimiser $F(\mu) = \sum_i \|(\mathbf{I} - \mathbf{P})(x_i - \mu)\|^2$. Notons $\mathbf{Q} =
\mathbf{I} - \mathbf{P}$, projecteur ($\mathbf{Q}^T = \mathbf{Q}$, $\mathbf{Q}^2 = \mathbf{Q}$). Le gradient
est

$$\nabla F(\mu) = -2\sum_i \mathbf{Q}^T\mathbf{Q}(x_i - \mu) = -2N\,\mathbf{Q}(\bar x - \mu).$$

$F$ est convexe (somme de carrés de fonctions affines) : ses minimiseurs sont les $\mu$ qui
annulent le gradient, c'est-à-dire $\mathbf{Q}(\mu - \bar x) = 0$, soit $\mu - \bar x \in
\operatorname{Im}(\mathbf{V}_q)$.